# 01 · Data Understanding

**Project:** Regional Sales Performance Analysis — RetailCo
**Goal of this notebook:** get to know the raw data *before* changing anything.

> You cannot clean what you have not looked at. Every cleaning decision in notebook 02 comes from something we discover here.

**Input:** `data/raw/sales_raw.csv` (the messy export)
**Output:** a list of data-quality problems → handed to `02_data_cleaning.ipynb`

---

## Part 1 — The basics, from zero

| Term | Plain-English meaning | Example in this project |
|---|---|---|
| **Dataset** | A table of recorded facts | Every product RetailCo sold, 2021–2024 |
| **Row** (record, observation) | One single fact | One product on one order: *"2 reams of paper on order US-2023-100926"* |
| **Column** (field, variable) | One property recorded for every row | `Sales`, `Region`, `Order Date` |
| **Grain** | What exactly *one row* represents | One **order line** (one product within one order) — *not* one order |
| **Feature** | A column used to describe or explain something | `Discount`, `Region`, `Category` |
| **Business metric (target)** | The number management cares about | `Sales`, `Profit`, and ratios built from them |
| **Categorical data** | Values are labels/groups; maths on them makes no sense | `Region`, `Segment`, `Category` |
| **Numerical data** | Values are quantities you can add or average | `Sales`, `Quantity`, `Profit` |
| **Datetime data** | Values are points in time | `Order Date`, `Ship Date` |

**Why the grain matters:** because one order can have several lines, *counting rows ≠ counting orders*. To count orders we must count **distinct** `Order ID`s. Mixing these up is one of the most common analyst mistakes.

## Part 2 — Setup

We import **pandas** (tables), **numpy** (fast maths) and our own helper module. `sys.path` is extended so the notebook can import code from `src/` — the same code the automated pipeline uses, so notebook and pipeline can never disagree.

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd

from utils import RAW_FILE, STATE_TO_REGION, VALID_REGIONS, VALID_CATEGORIES

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

### Load the raw file

**What:** `pd.read_csv()` reads a CSV (comma-separated text file) into a **DataFrame** — pandas' name for a table.
**Why:** everything else works on this DataFrame.
**Note:** we deliberately do **not** tell pandas the column types yet. Letting pandas guess, and then seeing where the guess goes wrong, is itself a data-quality check.

In [2]:
df = pd.read_csv(RAW_FILE)
print("Loaded:", RAW_FILE.name)

Loaded: sales_raw.csv


---
## Part 3 — First look

### `df.head()` — first 5 rows
1. **What it does:** shows the top 5 rows.
2. **Why:** a quick sanity check that the file loaded correctly (right columns, values in the right place).
3. **What we expect to learn:** what a typical row looks like.
4. **Effect on next step:** confirms the grain (one product per row) and the column names we will use everywhere.

In [3]:
df.head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,US-2024-102429,2024-08-07,2024-08-12,Second Class,GM-10648,Grace Martin F.,Consumer,United States,Baltimore,Maryland,East,OFF-PA-1012,Office Supplies,Paper,Premium Paper Pack H231,71.02,6,0.3,17.93
1,US-2024-103189,2024-03-24,2024-03-26,First Class,MH-10171,Maya Hughes,Consumer,United States,Richmond,Virginia,South,OFF-AR-1001,Office Supplies,Art,Fine-Tip Pack M493,17.71,3,0.1,4.36
2,US-2022-102079,2022-11-08,2022-11-08,Same Day,LR-10338,Lena Reddy,Consumer,United States,Newark,New Jersey,East,TEC-CO-1001,Technology,Copiers,Colour Copier Pro V730,4338.15,3,0.0,1748.86
3,US-2022-101355,2022-12-02,2022-12-07,Standard Class,IC-10375,Isla Clark,Corporate,United States,Boston,Massachusetts,East,OFF-PA-1018,Office Supplies,Paper,Recycled Paper Pack K309,27.86,3,0.2,9.43
4,US-2024-101404,2024-07-19,2024-07-24,Standard Class,LC-10282,Leo Chen,Corporate,United States,Philadelphia,Pennsylvania,East,OFF-BI-1004,Office Supplies,Binders,Heavy-Duty Binder T340,202.8,10,0.0,73.07


**Observation:** each row is one product within an order (`Order ID` + `Product ID`). Money columns are `Sales` and `Profit`; `Discount` is stored as a fraction (0.2 = 20%).

### `df.tail()` — last 5 rows
**Why:** problems often hide at the end of a file (a totals row, a half-written last line). Checking both ends is cheap insurance.

In [4]:
df.tail()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
11562,US-2024-104204,2024-12-18,2024-12-24,Standard Class,MR-10241,Mateo Rossi,Consumer,United States,Los Angeles,California,West,OFF-EN-1000,Office Supplies,Envelopes,Kraft Envelopes C952,58.14,5,0.4,-6.28
11563,US-2022-100250,2022-12-16,2022-12-17,First Class,LJ-10302,Lucas Johnson,Home Office,United States,Milwaukee,Wisconsin,Central,OFF-BI-1005,Office Supplies,Binders,Heavy-Duty Binder W184,92.7,10,0.0,37.78
11564,US-2022-100747,2022-12-30,2022-12-30,Same Day,EB-10716,Emma Brown,Consumer,United States,Detroit,Michigan,Central,OFF-PA-1015,Office Supplies,Paper,Premium Paper Ream F312,33.39,7,0.4,-2.18
11565,US-2021-100973,2021-06-08,2021-06-12,Standard Class,EC-10610,Emma Chen G.,Consumer,United States,Chicago,Illinois,Central,TEC-PH-1001,Technology,Phones,Nova Smartphone F319,532.01,4,0.4,-159.78
11566,US-2023-103121,2023-09-29,2023-10-06,Standard Class,OM-10379,Olivia Murphy,Corporate,United States,Richmond,Virginia,South,OFF-PA-1010,Office Supplies,Paper,Laser Paper Ream W159,35.04,2,0.2,9.53


### `df.shape` — size of the table
1. **What:** returns `(number of rows, number of columns)`.
2. **Why:** this is the **"before" number**. Every row removed in cleaning must be explained relative to it.
3. **Expect:** ~11.5k rows and 19 columns.
4. **Next step:** recorded as *Raw rows* in the cleaning log.

In [5]:
rows, cols = df.shape
print(f"Rows: {rows:,}   Columns: {cols}")

Rows: 11,567   Columns: 19


### `df.columns` — the column names
**Why:** code refers to columns by exact name. A hidden space (`"Sales "`) would break every query. Listing them catches that.

In [6]:
list(df.columns)

['Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

### `df.info()` and `df.dtypes` — data types and non-null counts
1. **What:** for each column, the data type pandas assigned and how many values are *not* missing.
2. **Why:** the **type decides what we can do**. You cannot sum text or sort dates stored as text correctly.
3. **Expect:** `Sales` should be a number (`float64`). Dates will be text (`object`/`str`) because CSV has no date type.
4. **Next step:** any column with the wrong type goes on the cleaning list.

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11567 entries, 0 to 11566
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Order ID       11567 non-null  str    
 1   Order Date     11567 non-null  str    
 2   Ship Date      11567 non-null  str    
 3   Ship Mode      11567 non-null  str    
 4   Customer ID    11567 non-null  str    
 5   Customer Name  11542 non-null  str    
 6   Segment        11567 non-null  str    
 7   Country        11567 non-null  str    
 8   City           11567 non-null  str    
 9   State          11567 non-null  str    
 10  Region         11532 non-null  str    
 11  Product ID     11567 non-null  str    
 12  Category       11567 non-null  str    
 13  Sub-Category   11567 non-null  str    
 14  Product Name   11547 non-null  str    
 15  Sales          11556 non-null  str    
 16  Quantity       11567 non-null  int64  
 17  Discount       11567 non-null  float64
 18  Profit         11

In [8]:
df.dtypes

Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales                str
Quantity           int64
Discount         float64
Profit           float64
dtype: object

**🚩 Finding 1 — `Sales` is stored as text, not a number.**
A numeric column only becomes text when *at least one* value is not a valid number. Let's find those values.

In [9]:
as_number = pd.to_numeric(df["Sales"], errors="coerce")   # invalid -> NaN
not_numeric = df.loc[as_number.isna() & df["Sales"].notna(), "Sales"]
print(f"{len(not_numeric)} Sales values are not plain numbers. Examples:")
print(not_numeric.head(8).tolist())

30 Sales values are not plain numbers. Examples:
['$301.39', '$1,277.10', '$231.18', '$750.82', '$584.48', '$17.99', '$28.18', '$91.44']


**Explanation:** some values were exported with a currency symbol and thousands separator (`"$1,277.10"`). They are *real* sales — we must convert them, not delete them.

**Also:** `Order Date` and `Ship Date` are text. We will convert them to real dates in cleaning, which will also reveal any impossible dates.

---
### `df.describe()` — summary statistics for numeric columns
1. **What:** count, mean, standard deviation, min, quartiles (25%, 50% = median, 75%) and max.
2. **Why:** the fastest way to spot impossible values — a negative quantity, a discount of 2000%.
3. **Expect:** `Quantity` ≥ 1, `Discount` between 0 and 1, `Profit` can be negative (losses are real).
4. **Next step:** anything outside these business rules becomes a validation rule.

In [10]:
df.describe().round(2)

,Quantity,Discount,Profit
count,11567.00,11567.00,11560.00
mean,4.94,0.17,39.34
std,2.84,0.83,182.62
min,-2.00,0.00,-2565.38
25%,2.00,0.00,3.98
50%,5.00,0.10,20.52
75%,7.00,0.20,65.91
max,10.00,40.00,1762.48


**🚩 Finding 2 — impossible values.**
* `Quantity` has a **minimum ≤ 0** → you cannot sell zero or minus two chairs.
* `Discount` has a **maximum far above 1** → a discount above 100% means RetailCo paid customers to take products. More likely someone typed `20` meaning 20%.
* `Profit` minimum is negative — this is **expected**: a heavily discounted sale can lose money. *Negative profit is information, not an error.*

Let's quantify these.

In [11]:
sales_num = pd.to_numeric(df["Sales"].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce")
checks = pd.Series({
    "Sales <= 0":           int((sales_num <= 0).sum()),
    "Quantity <= 0":        int((df["Quantity"] <= 0).sum()),
    "Discount > 1":         int((df["Discount"] > 1).sum()),
    "Profit < 0 (valid!)":  int((df["Profit"] < 0).sum()),
})
checks.to_frame("rows")

,rows
Sales <= 0,8
Quantity <= 0,6
Discount > 1,18
Profit < 0 (valid!),1926


---
## Part 4 — Missing values

1. **What:** `df.isna().sum()` counts empty cells per column.
2. **Why:** missing values silently change results — `sum()` skips them, `count()` ignores them, and a missing `Region` means that sale disappears from every regional chart.
3. **Expect:** most columns complete; a few gaps in names, region and money columns.
4. **Next step:** for each column we decide: **rebuild**, **keep as missing**, or **remove** — see notebook 02.

In [12]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
})
missing[missing["missing"] > 0].sort_values("missing", ascending=False)

,missing,missing_%
Region,35,0.30
Customer Name,25,0.22
Product Name,20,0.17
Sales,11,0.10
Profit,7,0.06


**🚩 Finding 3 — missing values, and what each one means:**

| Column | Can it be rebuilt? | How |
|---|---|---|
| `Region` | ✅ Yes | Every **State** belongs to exactly one region (verified below) |
| `Customer Name` | ✅ Yes | The same `Customer ID` appears on other rows with the name |
| `Product Name` | ✅ Yes | Same idea, via `Product ID` |
| `Sales`, `Profit` | ❌ No | The data has no list price or cost column to recompute them from |

**Proving that State → Region is reliable** (if a state appeared in two regions, rebuilding would be guesswork):

In [13]:
clean_region = df["Region"].str.strip().str.lower()
regions_per_state = df.assign(r=clean_region).dropna(subset=["r"]).groupby("State")["r"].unique()
regions_per_state[regions_per_state.apply(len) > 1]

State
Illinois    [central, centrl]
Texas       [central, centrl]
Name: r, dtype: object

Only **Texas** and **Illinois** show a second "region" — and it is the typo `centrl`, not a genuinely different region. So each state maps to exactly one region and rebuilding `Region` from `State` is safe.

---
## Part 5 — Duplicates

There are **two very different kinds** of "duplicate" here. Confusing them would be a serious mistake.

### 5a. Exact duplicate rows
1. **What:** `df.duplicated()` marks a row that is identical, in *every* column, to an earlier row.
2. **Why:** a line exported twice would double-count its revenue.
3. **Next step:** remove the extra copies (keep the first).

In [14]:
exact_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes}")

Exact duplicate rows: 70


### 5b. Repeated Order IDs — **normal, not an error**
One order can contain several products, so `Order ID` *should* repeat. Deleting "duplicate" Order IDs would delete real sales.

In [15]:
lines_per_order = df.drop_duplicates().groupby("Order ID")["Product ID"].nunique()
lines_per_order.value_counts().sort_index().rename_axis("lines in order").to_frame("number of orders")

,number of orders
lines in order,
1,1994
2,1349
3,944
4,592
5,321


Most orders have more than one line. **Conclusion:** the correct uniqueness key is **`Order ID` + `Product ID`**, not `Order ID` alone.

---
## Part 6 — Unique values and category consistency

1. **What:** `nunique()` counts distinct values; `value_counts()` lists them with frequencies.
2. **Why:** categorical columns should contain a small, fixed set of labels. `"East"`, `"east"` and `" East"` look the same to a human but are **three different groups** to a computer — the East region's sales would be split three ways.
3. **Expect:** 4 regions, 3 categories, 3 segments, 4 ship modes.

In [16]:
df.nunique().to_frame('unique values')

,unique values
Order ID,5200
Order Date,1360
Ship Date,1389
Ship Mode,4
Customer ID,741
Customer Name,741
Segment,3
Country,1
City,64
State,28


In [17]:
df["Region"].value_counts(dropna=False)

Region
West         3818
East         3111
Central      2543
South        2000
NaN            35
east           12
west            7
WEST            6
 West           6
CENTRAL         5
EAST            5
 Central        4
south           4
Centrl          3
central         3
 South          2
 East           2
SOUTH           1
Name: count, dtype: int64

In [18]:
df["Category"].value_counts(dropna=False)

Category
Office Supplies    6823
Furniture          2476
Technology         2223
Technology           13
Tech                 11
FURNITURE            11
office supplies      10
Name: count, dtype: int64

**🚩 Finding 4 — inconsistent labels.** `Region` has many spellings for 4 real regions (case differences, extra spaces, the typo `Centrl`). `Category` has `Tech`, `FURNITURE`, `office supplies` and a trailing-space `Technology `.

### Leading / trailing spaces in other text columns

In [19]:
text_cols = df.select_dtypes(include=["object", "string"]).columns
spaces = {c: int((df[c].dropna() != df[c].dropna().str.strip()).sum()) for c in text_cols}
pd.Series(spaces)[lambda s: s > 0].to_frame("values with extra spaces")

,values with extra spaces
City,40
Region,14
Category,13


### Dates
`pd.to_datetime(..., errors="coerce")` converts text to dates and turns anything that is not a real calendar date into `NaT` ("Not a Time").

In [20]:
order_dt = pd.to_datetime(df["Order Date"], format="%Y-%m-%d", errors="coerce")
ship_dt = pd.to_datetime(df["Ship Date"], format="%Y-%m-%d", errors="coerce")
print("Order dates that are not real dates:", df.loc[order_dt.isna(), "Order Date"].tolist())
print("Ship date earlier than order date   :", int((ship_dt < order_dt).sum()), "rows")
print("Date range:", order_dt.min().date(), "to", order_dt.max().date())

Order dates that are not real dates: ['not recorded', '2023-02-30', 'not recorded', '2023-02-30', '2022-13-05']
Ship date earlier than order date   : 7 rows
Date range: 2021-01-01 to 2024-12-31


**🚩 Finding 5 — invalid dates.** `2023-02-30` (February has no 30th), `2022-13-05` (no 13th month) and `not recorded`. A few rows also ship *before* they were ordered, which is impossible.

---
## Part 7 — Summary: the data-quality issue list

This table is the **hand-over to notebook 02**. Each problem has a planned treatment and a reason.

In [21]:
issues = pd.DataFrame([
    ("Sales stored as text ($, commas)", len(not_numeric), "Convert to number", "Values are real sales, only badly formatted"),
    ("Exact duplicate rows", exact_dupes, "Remove extra copies", "Would double-count revenue"),
    ("Region missing", int(df["Region"].isna().sum()), "Rebuild from State", "State -> Region is one-to-one"),
    ("Region / Category spelling variants", int((~df["Region"].isin(VALID_REGIONS) & df["Region"].notna()).sum() + (~df["Category"].isin(VALID_CATEGORIES)).sum()), "Standardise", "Same group written differently"),
    ("Missing customer / product names", int(df["Customer Name"].isna().sum() + df["Product Name"].isna().sum()), "Rebuild from IDs", "IDs are complete and reliable"),
    ("Discount typed as percent (>1)", checks["Discount > 1"], "Divide by 100 if result is a valid level", "20 almost certainly means 20%"),
    ("Missing Sales / Profit", int(sales_num.isna().sum() - len(not_numeric) + df["Profit"].isna().sum()), "Remove", "Cannot be rebuilt; imputing would invent money"),
    ("Sales <= 0 or Quantity <= 0", checks["Sales <= 0"] + checks["Quantity <= 0"], "Remove", "Impossible for a sale; returns live elsewhere"),
    ("Invalid order date", int(order_dt.isna().sum()), "Remove", "Row cannot be placed in time"),
    ("Ship date before order date", int((ship_dt < order_dt).sum()), "Blank Ship Date, keep row", "The sale is real; only shipping info is wrong"),
], columns=["Issue", "Rows (raw, before de-duplication)", "Planned treatment", "Reason"])
issues

,Issue,"Rows (raw, before de-duplication)",Planned treatment,Reason
0,"Sales stored as text ($, commas)",30,Convert to number,"Values are real sales, only badly formatted"
1,Exact duplicate rows,70,Remove extra copies,Would double-count revenue
2,Region missing,35,Rebuild from State,State -> Region is one-to-one
3,Region / Category spelling variants,105,Standardise,Same group written differently
4,Missing customer / product names,45,Rebuild from IDs,IDs are complete and reliable
5,Discount typed as percent (>1),18,Divide by 100 if result is a valid level,20 almost certainly means 20%
6,Missing Sales / Profit,-12,Remove,Cannot be rebuilt; imputing would invent money
7,Sales <= 0 or Quantity <= 0,14,Remove,Impossible for a sale; returns live elsewhere
8,Invalid order date,5,Remove,Row cannot be placed in time
9,Ship date before order date,7,"Blank Ship Date, keep row",The sale is real; only shipping info is wrong


**Note on counts:** these are counted on the *raw* file, which still contains the 70 duplicate rows. A few problem rows are themselves duplicated, so the counts in notebook 02 (which removes duplicates first) can be slightly lower. Both are correct for their stage.

**Next:** `02_data_cleaning.ipynb` applies each treatment and records the before/after numbers.